In [1]:
# ============================================================
# HYBRID TWO-STAGE SYSTEM - FINAL FIXED (KAGGLE READY)
# ============================================================

import os
os.environ['WANDB_DISABLED'] = 'true'

import json
import torch
from transformers import BertTokenizer, BertForSequenceClassification, AutoTokenizer
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
import numpy as np

print("="*60)
print("BUILDING HYBRID TWO-STAGE SYSTEM")
print("="*60)

# ============================================================
# STEP 1: Load Intent Classification Model (Stage 1)
# ============================================================

print("\n📥 Loading Stage 1: Intent Classification Model...")

INTENT_MODEL_PATH = "/kaggle/input/intent-classification-model"
INTENT_MAPPING_PATH = os.path.join(INTENT_MODEL_PATH, "intent_mapping.json")

required_files = [
    "config.json",
    "model.safetensors",
    "intent_mapping.json",
    "vocab.txt",
    "tokenizer_config.json"
]
missing = [f for f in required_files if not os.path.exists(os.path.join(INTENT_MODEL_PATH, f))]
if missing:
    raise FileNotFoundError(f"❌ Missing intent model files: {missing}")

# Load intent mapping
with open(INTENT_MAPPING_PATH, 'r', encoding='utf-8') as f:
    intent_to_id = json.load(f)
id_to_intent = {v: k for k, v in intent_to_id.items()}
num_labels = len(intent_to_id)
print(f"✅ Loaded {num_labels} intents")

# Load tokenizer and model
tokenizer_intent = AutoTokenizer.from_pretrained(INTENT_MODEL_PATH, local_files_only=True)
model_intent = BertForSequenceClassification.from_pretrained(
    INTENT_MODEL_PATH,
    num_labels=num_labels,
    local_files_only=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_intent = model_intent.to(device)
model_intent.eval()

print(f"✅ Intent Classification Model Loaded on {device}")

# ============================================================
# STEP 2: Load Semantic Retrieval Model (Stage 2)
# ============================================================

print("\n📥 Loading Stage 2: Semantic Retrieval Model...")

# ✅ Fix: Use correct subfolder inside your dataset
SEMANTIC_MODEL_PATH = "/kaggle/input/assamese-tourism-semantic-model/semantic_retrieval_model"

if os.path.exists(SEMANTIC_MODEL_PATH):
    try:
        model_semantic = SentenceTransformer(SEMANTIC_MODEL_PATH, device=device)
        use_semantic = True
        print(f"✅ Semantic Retrieval Model Loaded from {SEMANTIC_MODEL_PATH}")
    except Exception as e:
        print(f"⚠️ Error loading semantic model: {e}")
        print("   → Falling back to intent-only mode")
        model_semantic = None
        use_semantic = False
else:
    print("⚠️ Semantic model path not found.")
    use_semantic = False
    model_semantic = None

# ============================================================
# STEP 3: Load Knowledge Base
# ============================================================

print("\n📥 Loading Knowledge Base (Q&A dataset)...")

QA_DATASET_PATH = None
for root, dirs, files in os.walk('/kaggle/input/'):
    for file in files:
        if file.endswith('.json') and any(k in file.lower() for k in ['complete', 'qa', 'dataset', 'train']):
            QA_DATASET_PATH = os.path.join(root, file)
            break
    if QA_DATASET_PATH:
        break

if not QA_DATASET_PATH:
    raise FileNotFoundError("❌ Q&A dataset not found!")

with open(QA_DATASET_PATH, 'r', encoding='utf-8') as f:
    qa_dataset = json.load(f)

print(f"✅ Loaded {len(qa_dataset)} Q&A pairs")

# ============================================================
# STEP 4: Create Embeddings (if semantic model available)
# ============================================================

if use_semantic:
    print("\n🔄 Creating embeddings for all questions...")
    all_questions = [item['query'] for item in qa_dataset]
    all_embeddings = model_semantic.encode(all_questions, show_progress_bar=True)
    print(f"✅ Created embeddings for {len(all_questions)} questions")
else:
    all_embeddings = None
    print("⚠️ Skipping embeddings (semantic model not available)")

# ============================================================
# STEP 5: Define Hybrid Functions
# ============================================================

def classify_intent(query):
    inputs = tokenizer_intent(
        query,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model_intent(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        label = torch.argmax(probs, dim=-1).item()
    return id_to_intent[label], probs[0][label].item()

def semantic_search(query, intent, top_k=5):
    if not use_semantic or all_embeddings is None:
        matches = [i for i in qa_dataset if i.get('intent') == intent]
        return matches[:top_k]
    query_emb = model_semantic.encode(query)
    intent_filtered = [(i, item) for i, item in enumerate(qa_dataset) if item.get('intent') == intent]
    if not intent_filtered:
        return []
    indices = [i for i, _ in intent_filtered]
    emb_subset = all_embeddings[indices]
    sims = [(idx, 1 - cosine(query_emb, emb)) for idx, emb in zip(indices, emb_subset)]
    sims.sort(key=lambda x: x[1], reverse=True)
    results = []
    for idx, sim in sims[:top_k]:
        results.append({
            'query': qa_dataset[idx]['query'],
            'response': qa_dataset[idx]['response'],
            'intent': qa_dataset[idx]['intent'],
            'similarity': sim
        })
    return results

def hybrid_system(query, top_k=3):
    print(f"\n{'='*60}\nQUERY: {query}\n{'='*60}")
    intent, conf = classify_intent(query)
    print(f"🔍 Intent: {intent} ({conf:.2%})")
    results = semantic_search(query, intent, top_k=top_k)
    if not results:
        print("⚠️ No results found.")
        return
    print(f"✅ Found {len(results)} results:")
    for i, r in enumerate(results, 1):
        print(f"\n{i}. {r['query']}")
        print(f"→ {r['response'][:100]}...")
    print("\n✅ Best Answer:", results[0]['response'])

print("\n✅ Hybrid System Ready!")

# ============================================================
# STEP 6: TEST EXAMPLES
# ============================================================

print("\n🧪 TESTING SYSTEM\n" + "="*60)
test_queries = [
    "Kaziranga t keidin thakibo lage?",
    "Majuli kenekoie jabo pari?",
    "Best time to visit Kamakhya Temple?",
    "What to eat in Guwahati?",
    "Is Majuli safe for tourists?"
]

for query in test_queries:
    hybrid_system(query, top_k=3)
    print("-"*60)

print("\n🎉 All tests complete!")


2025-11-01 09:38:40.062906: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761989920.362752      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761989920.445884      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


BUILDING HYBRID TWO-STAGE SYSTEM

📥 Loading Stage 1: Intent Classification Model...
✅ Loaded 16 intents
✅ Intent Classification Model Loaded on cpu

📥 Loading Stage 2: Semantic Retrieval Model...


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Semantic Retrieval Model Loaded from /kaggle/input/assamese-tourism-semantic-model/semantic_retrieval_model

📥 Loading Knowledge Base (Q&A dataset)...
✅ Loaded 3081 Q&A pairs

🔄 Creating embeddings for all questions...


Batches:   0%|          | 0/97 [00:00<?, ?it/s]

...

✅ Found 3 results:

1. Ideal time for Kamakhya Temple?
→ Information about Kamakhya Temple regarding best time visit....

2. When to visit Kamakhya Temple?
→ Kamakhya Temple juar karone October to March best time. Winter months ideal for visiting....

3. Good time to go Kamakhya Temple?
→ Information about Kamakhya Temple for best time visit....

✅ Best Answer: Information about Kamakhya Temple regarding best time visit.
------------------------------------------------------------

QUERY: What to eat in Guwahati?
🔍 Intent: food_recommendation (96.05%)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Found 3 results:

1. What to eat in Guwahati?
→ Information about Guwahati regarding food recommendation....

2. Where to dine in Guwahati?
→ Information about Guwahati for food recommendation....

3. Where to eat near Guwahati?
→ Let me help you with tourism information about Assam....

✅ Best Answer: Information about Guwahati regarding food recommendation.
------------------------------------------------------------

QUERY: Is Majuli safe for tourists?
🔍 Intent: safety_emergency (96.26%)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Found 3 results:

1. Is Majuli safe for tourists?
→ Let me help you with tourism information about Assam....

2. Is Majuli Island safe for tourists?
→ Majuli Island safe for tourists. Emergency: Police 100, Ambulance 108. Follow safety guidelines....

3. Emergency contact for Majuli?
→ Let me help you with tourism information about Assam....

✅ Best Answer: Let me help you with tourism information about Assam.
------------------------------------------------------------

🎉 All tests complete!


In [2]:
print("\n🧪 TESTING SYSTEM\n" + "="*60)
test_queries = [
    "Sivasagar t keidin thakibo lage?",

]

for query in test_queries:
    hybrid_system(query, top_k=3)
    print("-"*60)

print("\n🎉 All tests complete!")


🧪 TESTING SYSTEM

QUERY: Sivasagar t keidin thakibo lage?
🔍 Intent: best_time_visit (93.70%)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Found 3 results:

1. Sivasagar keitiya visit koribo lage?
→ Information about Sivasagar for best time visit....

2. Sivasagar t keitiya jabo bhal?
→ Information about Sivasagar for best time visit....

3. Climate wise best time Sivasagar?
→ Information about Sivasagar for best time visit....

✅ Best Answer: Information about Sivasagar for best time visit.
------------------------------------------------------------

🎉 All tests complete!
